# M3T2 - Ejercicio guiado con kafka-spark-streaming-example

Este cuaderno descarga el repositorio [`dbusteed/kafka-spark-streaming-example`](https://github.com/dbusteed/kafka-spark-streaming-example) y prepara una versión ampliada del ejercicio propuesto en la lección.

El repositorio original implementa una prueba de concepto con Kafka, Spark Streaming, Hadoop e Hive: un productor publica tweets en Kafka, Spark consume el topic `tweets`, calcula métricas simples y persiste los resultados en Hive.

Este cuaderno automatiza las tareas de mejora: productor JSON con timestamp, esquema Hive ampliado, transformadores mejorados, checkpoints, salida Parquet, cuarentena, monitorización y versión alternativa con Structured Streaming.

## 1. Preparación

Requisitos esperados si se quiere ejecutar el flujo completo:

- Java 8 o compatible con la versión de Spark utilizada.
- Kafka levantado en `localhost:9092`.
- Hadoop/HDFS y Hive si se quiere reproducir el sink Hive original.
- Spark instalado y accesible mediante `spark-submit`.

Si no tienes Kafka/Hive/Spark levantados, el cuaderno igualmente genera los ficheros de mejora y los comandos necesarios.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORKDIR = Path.cwd()
REPO_URL = "https://github.com/dbusteed/kafka-spark-streaming-example.git"
REPO_DIR = WORKDIR / "kafka-spark-streaming-example"
FILES_DIR = REPO_DIR / "files"
LAB_DIR = REPO_DIR / "m3t2_lab"

print("Directorio de trabajo:", WORKDIR)
print("Repositorio destino:", REPO_DIR)

## 2. Descargar el repositorio

La celda clona el repositorio si no existe. Si ya está clonado, muestra su estado.

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("El repositorio ya existe, no se clona de nuevo.")

subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"], check=False)
print("Ficheros principales:")
for path in sorted(FILES_DIR.glob("*.py")):
    print("-", path.relative_to(REPO_DIR))

## 3. Instalar dependencias Python

Estas dependencias permiten ejecutar el productor y, si procede, el transformador con PySpark. Si el entorno ya las tiene instaladas, la celda no debería cambiar nada relevante.

In [ ]:
# Descomenta si necesitas instalar dependencias en el kernel actual.
# %pip install kafka-python tweepy pyspark==3.5.1

## 4. Crear estructura de trabajo para las mejoras

In [ ]:
LAB_DIR.mkdir(exist_ok=True)
(LAB_DIR / "output").mkdir(exist_ok=True)
(LAB_DIR / "checkpoint").mkdir(exist_ok=True)
(LAB_DIR / "quarantine").mkdir(exist_ok=True)

print("Laboratorio preparado en:", LAB_DIR)

## 5. Mejora 1 y 3: productor simulado con timestamp y JSON

Se genera un productor alternativo que publica mensajes JSON en el topic `tweets`. Incluye `event_id`, `event_time`, `source`, texto, hashtags y menciones.

In [ ]:
producer_code = r'''#!/usr/bin/env python3
import json
import random
import time
import uuid
from datetime import datetime, timezone

from kafka import KafkaProducer

TOPIC = "tweets"
BOOTSTRAP_SERVERS = "localhost:9092"
SLEEP_SECONDS = 1

WORDS = [
    "spark", "kafka", "streaming", "hive", "python", "data", "pipeline",
    "latency", "checkpoint", "topic", "producer", "consumer", "parquet"
]
HASHTAGS = ["#bigdata", "#spark", "#kafka", "#streaming", "#python"]
MENTIONS = ["@data_team", "@spark_user", "@kafka_lab"]

def build_tweet():
    word_count = random.randint(6, 16)
    tokens = random.choices(WORDS, k=word_count)
    if random.random() < 0.7:
        tokens.append(random.choice(HASHTAGS))
    if random.random() < 0.4:
        tokens.append(random.choice(MENTIONS))

    text = " ".join(tokens)
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    return {
        "event_id": str(uuid.uuid4()),
        "event_time": now,
        "source": "m3t2_improved_fake_tweet_stream",
        "text": text,
        "hashtags": [token for token in tokens if token.startswith("#")],
        "mentions": [token for token in tokens if token.startswith("@")]
    }

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    value_serializer=lambda value: json.dumps(value).encode("utf-8"),
    key_serializer=lambda value: value.encode("utf-8")
)

print(f"Publicando eventos JSON en topic {TOPIC}...")
while True:
    event = build_tweet()
    producer.send(TOPIC, key=event["event_id"], value=event)
    producer.flush()
    print(event)
    time.sleep(SLEEP_SECONDS)
'''

producer_path = LAB_DIR / "improved_fake_tweet_stream.py"
producer_path.write_text(producer_code, encoding="utf-8")
producer_path.chmod(0o755)
print("Productor generado:", producer_path)

## 6. Mejora 2: esquema Hive ampliado

Se crea un script SQL con una tabla más rica que la tabla original. Añade `source`, timestamps y métricas adicionales.

In [ ]:
hive_sql = r'''CREATE TABLE IF NOT EXISTS tweets_enriched (
  event_id STRING,
  source STRING,
  event_time STRING,
  processed_time STRING,
  text STRING,
  words INT,
  length INT,
  hashtag_count INT,
  mention_count INT,
  unique_words INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '|'
STORED AS TEXTFILE;

CREATE TABLE IF NOT EXISTS tweets_quarantine (
  raw_value STRING,
  error_reason STRING,
  processed_time STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '|'
STORED AS TEXTFILE;
'''

hive_sql_path = LAB_DIR / "hive_schema_improved.sql"
hive_sql_path.write_text(hive_sql, encoding="utf-8")
print("SQL Hive generado:", hive_sql_path)
print(hive_sql)

## 7. Mejoras 4, 5, 7, 9 y 10: transformador Spark Streaming clásico

Este transformador mantiene la aproximación del repositorio original basada en DStreams. Consume JSON desde Kafka, valida campos, calcula métricas, usa checkpoint y separa registros inválidos.

In [ ]:
classic_transformer_code = r'''#!/usr/bin/env python3
import json
from datetime import datetime, timezone

from pyspark import SparkContext
from pyspark.streaming import StreamingContext
from pyspark.streaming.kafka import KafkaUtils

APP_NAME = "M3T2ImprovedClassicTransformer"
TOPIC = "tweets"
ZOOKEEPER = "localhost:2181"
CHECKPOINT_DIR = "m3t2_lab/checkpoint/classic_transformer"
PARQUET_OUTPUT = "m3t2_lab/output/tweets_enriched_parquet"
QUARANTINE_OUTPUT = "m3t2_lab/quarantine/tweets_invalid"

def parse_event(raw):
    processed_time = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    try:
        event = json.loads(raw)
        text = event.get("text", "")
        if not event.get("event_id"):
            raise ValueError("missing event_id")
        if not text.strip():
            raise ValueError("empty text")

        tokens = text.split()
        hashtags = [token for token in tokens if token.startswith("#")]
        mentions = [token for token in tokens if token.startswith("@")]
        return ("valid", {
            "event_id": event.get("event_id"),
            "source": event.get("source", "unknown"),
            "event_time": event.get("event_time"),
            "processed_time": processed_time,
            "text": text.replace("|", " "),
            "words": len(tokens),
            "length": len(text),
            "hashtag_count": len(hashtags),
            "mention_count": len(mentions),
            "unique_words": len(set(token.lower() for token in tokens))
        })
    except Exception as exc:
        return ("invalid", {
            "raw_value": raw.replace("|", " "),
            "error_reason": str(exc).replace("|", " "),
            "processed_time": processed_time
        })

def save_valid_rdd(rdd):
    if rdd.isEmpty():
        return
    spark = SparkContext.getOrCreate()._jvm
    # En el ejercicio se recomienda adaptar esta parte a HiveContext/SparkSession según versión.
    print("Registros validos en microbatch:", rdd.count())
    rdd.map(lambda x: "|".join(str(x.get(k, "")) for k in [
        "event_id", "source", "event_time", "processed_time", "text", "words",
        "length", "hashtag_count", "mention_count", "unique_words"
    ])).saveAsTextFile(PARQUET_OUTPUT + "_text_batch")

def save_invalid_rdd(rdd):
    if rdd.isEmpty():
        return
    print("Registros invalidos en microbatch:", rdd.count())
    rdd.map(lambda x: "|".join([x.get("raw_value", ""), x.get("error_reason", ""), x.get("processed_time", "")])) \
       .saveAsTextFile(QUARANTINE_OUTPUT + "_batch")

sc = SparkContext(appName=APP_NAME)
ssc = StreamingContext(sc, 5)
ssc.checkpoint(CHECKPOINT_DIR)

stream = KafkaUtils.createStream(ssc, ZOOKEEPER, "m3t2-classic-consumer", {TOPIC: 1})
parsed = stream.map(lambda kv: kv[1]).map(parse_event)

valid = parsed.filter(lambda item: item[0] == "valid").map(lambda item: item[1])
invalid = parsed.filter(lambda item: item[0] == "invalid").map(lambda item: item[1])

valid.foreachRDD(save_valid_rdd)
invalid.foreachRDD(save_invalid_rdd)

ssc.start()
ssc.awaitTermination()
'''

classic_transformer_path = LAB_DIR / "improved_transformer_classic.py"
classic_transformer_path.write_text(classic_transformer_code, encoding="utf-8")
classic_transformer_path.chmod(0o755)
print("Transformador Spark Streaming clásico generado:", classic_transformer_path)

## 8. Mejoras 8 y 11: versión con Structured Streaming y salida Parquet

Esta versión moderniza el ejemplo: usa `readStream.format('kafka')`, parsea JSON con esquema, escribe datos válidos en Parquet y envía eventos inválidos a cuarentena.

In [ ]:
structured_transformer_code = r'''#!/usr/bin/env python3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, length, size, split, expr, to_json, struct
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, TimestampType

BOOTSTRAP_SERVERS = "localhost:9092"
TOPIC = "tweets"
CHECKPOINT_VALID = "m3t2_lab/checkpoint/structured_valid"
CHECKPOINT_INVALID = "m3t2_lab/checkpoint/structured_invalid"
VALID_OUTPUT = "m3t2_lab/output/tweets_enriched_parquet"
INVALID_OUTPUT = "m3t2_lab/quarantine/tweets_invalid_parquet"

spark = (SparkSession.builder
    .appName("M3T2StructuredStreamingTweets")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("source", StringType(), True),
    StructField("text", StringType(), True),
    StructField("hashtags", ArrayType(StringType()), True),
    StructField("mentions", ArrayType(StringType()), True)
])

raw = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .option("maxOffsetsPerTrigger", "1000")
    .load())

from pyspark.sql.functions import from_json

parsed = (raw
    .selectExpr("CAST(key AS STRING) AS kafka_key", "CAST(value AS STRING) AS raw_value", "timestamp AS kafka_timestamp")
    .select("kafka_key", "raw_value", "kafka_timestamp", from_json(col("raw_value"), schema).alias("event"))
    .select("kafka_key", "raw_value", "kafka_timestamp", "event.*")
    .withColumn("processed_time", current_timestamp()))

valid = (parsed
    .filter(col("event_id").isNotNull() & col("text").isNotNull() & (length(col("text")) > 0))
    .withColumn("words", size(split(col("text"), " ")))
    .withColumn("length", length(col("text")))
    .withColumn("hashtag_count", size(col("hashtags")))
    .withColumn("mention_count", size(col("mentions")))
    .withColumn("unique_words", expr("size(array_distinct(split(lower(text), ' ')))")))

invalid = (parsed
    .filter(col("event_id").isNull() | col("text").isNull() | (length(col("text")) == 0))
    .select("raw_value", "processed_time"))

valid_query = (valid.writeStream
    .format("parquet")
    .option("path", VALID_OUTPUT)
    .option("checkpointLocation", CHECKPOINT_VALID)
    .outputMode("append")
    .start())

invalid_query = (invalid.writeStream
    .format("parquet")
    .option("path", INVALID_OUTPUT)
    .option("checkpointLocation", CHECKPOINT_INVALID)
    .outputMode("append")
    .start())

spark.streams.awaitAnyTermination()
'''

structured_transformer_path = LAB_DIR / "improved_transformer_structured.py"
structured_transformer_path.write_text(structured_transformer_code, encoding="utf-8")
structured_transformer_path.chmod(0o755)
print("Transformador Structured Streaming generado:", structured_transformer_path)

## 9. Mejora 6: topic con varias particiones

La siguiente celda genera los comandos para crear el topic `tweets` con varias particiones. Ejecútalos en una terminal donde `KAFKA_HOME` apunte a la instalación de Kafka.

In [ ]:
topic_commands = r'''# Crear topic con varias particiones
$KAFKA_HOME/bin/kafka-topics.sh --create \
  --bootstrap-server localhost:9092 \
  --replication-factor 1 \
  --partitions 3 \
  --topic tweets

# Comprobar descripcion del topic
$KAFKA_HOME/bin/kafka-topics.sh --describe \
  --bootstrap-server localhost:9092 \
  --topic tweets
'''

topic_commands_path = LAB_DIR / "kafka_topic_commands.sh"
topic_commands_path.write_text(topic_commands, encoding="utf-8")
print(topic_commands)

## 10. Comandos de ejecución

Se genera un fichero con comandos para arrancar el productor y los transformadores. Ajusta rutas de Kafka, Spark y JARs según tu instalación.

In [ ]:
run_commands = f'''#!/usr/bin/env bash
set -e

# Terminal 1: Zookeeper
# $KAFKA_HOME/bin/zookeeper-server-start.sh $KAFKA_HOME/config/zookeeper.properties

# Terminal 2: Kafka broker
# $KAFKA_HOME/bin/kafka-server-start.sh $KAFKA_HOME/config/server.properties

# Terminal 3: Crear topic
# bash {topic_commands_path}

# Terminal 4: Hive Metastore
# hive --services metastore

# Terminal 5: Crear tablas Hive
# hive -f {hive_sql_path}

# Terminal 6: Productor JSON mejorado
python3 {producer_path}

# Terminal 7A: Transformador clasico Spark Streaming
# spark-submit --jars spark-streaming-kafka-0-8-assembly_2.11-2.4.3.jar {classic_transformer_path}

# Terminal 7B: Transformador Structured Streaming moderno
# spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1 {structured_transformer_path}
'''

run_commands_path = LAB_DIR / "run_exercise_commands.sh"
run_commands_path.write_text(run_commands, encoding="utf-8")
run_commands_path.chmod(0o755)
print("Comandos generados en:", run_commands_path)
print(run_commands)

## 11. Validaciones locales de los ficheros generados

Estas validaciones no requieren Kafka ni Spark: comprueban que los ficheros se han generado y que los scripts Python compilan.

In [ ]:
generated_files = [
    producer_path,
    hive_sql_path,
    classic_transformer_path,
    structured_transformer_path,
    topic_commands_path,
    run_commands_path,
]

for path in generated_files:
    assert path.exists(), f"No existe {path}"
    print("OK", path.relative_to(REPO_DIR))

subprocess.run(["python3", "-m", "py_compile", str(producer_path)], check=True)
subprocess.run(["python3", "-m", "py_compile", str(classic_transformer_path)], check=True)
subprocess.run(["python3", "-m", "py_compile", str(structured_transformer_path)], check=True)
print("Scripts Python compilados correctamente.")

## 12. Mejora 12: documentación de decisiones de laboratorio vs producción

La celda genera una memoria breve que puedes completar con capturas, métricas y conclusiones.

In [ ]:
report = r'''# Memoria del ejercicio M3T2

## Repositorio base

- https://github.com/dbusteed/kafka-spark-streaming-example

## Cambios implementados

1. Productor simulado con `event_id`, `event_time`, `source` y payload JSON.
2. Esquema Hive ampliado con timestamps, métricas y tabla de cuarentena.
3. Transformador clásico con validación, checkpoint y separación de registros inválidos.
4. Transformador moderno con Structured Streaming y salida Parquet.
5. Topic `tweets` propuesto con varias particiones.
6. Monitorización básica mediante conteos por microbatch y checkpoints.

## Decisiones válidas para laboratorio

- Un solo broker Kafka.
- Factor de replicación 1.
- Hive con Derby local.
- Scripts lanzados manualmente en terminales separadas.

## Cambios necesarios para producción

- Clúster Kafka con varias réplicas y seguridad activada.
- Checkpoints en almacenamiento distribuido fiable.
- Esquemas versionados con Schema Registry o contrato equivalente.
- Sinks idempotentes y estrategia clara de reprocesamiento.
- Monitorización de lag, throughput, errores de parseo y fallos de escritura.

## Evidencias a incluir

- Captura del topic `tweets` creado.
- Captura del productor publicando eventos.
- Captura de `spark-submit` procesando microbatches.
- Consulta Hive o lectura Parquet con registros procesados.
- Métricas de número de eventos procesados por minuto.
'''

report_path = LAB_DIR / "MEMORIA_EJERCICIO_M3T2.md"
report_path.write_text(report, encoding="utf-8")
print("Memoria generada:", report_path)
print(report)